Training notebook

In [1]:
import pandas as pd

In [ ]:
df=pd.read_csv('dataset/synthetic_logs.csv')

df.head()



,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert
...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert
2407,2025-08-03 03:07:47,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert


In [7]:
df.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI', 'LegacyCRM'], dtype=object)

In [8]:
df.target_label.unique()

array(['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
       'System Notification', 'Resource Usage', 'User Action',
       'Workflow Error', 'Deprecation Warning'], dtype=object)

In [33]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
import numpy as np

#load pre trained sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

#generate embeddings for the logs
embeddings = model.encode(df['log_message'].tolist())

#perform DBSCAN Clustering
dbscan= DBSCAN(eps=0.2, min_samples=1, metric='cosine')
clusters=dbscan.fit_predict(embeddings)

#add cluster labels to the dataframe
df['cluster']=clusters

#visualize the clusters
df['cluster']



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0       0
1       1
2       2
3       0
4       0
       ..
2405    0
2406    7
2407    0
2408    1
2409    7
Name: cluster, Length: 2410, dtype: int64

In [37]:
df['cluster'].unique()

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135])

In [36]:
df[df['cluster'] == 5]['log_message'].head()


8     nova.compute.claims [req-a07ac654-8e81-416d-bf...
26    nova.compute.claims [req-d6986b54-3735-4a42-90...
40    nova.compute.claims [req-72b4858f-049e-49e1-b3...
58    nova.compute.claims [req-5c8f52bd-8e3c-41f0-95...
61    nova.compute.claims [req-d38f479d-9bb9-4276-96...
Name: log_message, dtype: object

In [40]:
df[df.cluster==7]

,timestamp,source,log_message,target_label,complexity,cluster
13,8/4/2025 19:57,ThirdPartyAPI,Multiple bad login attempts detected on user 8...,Security Alert,bert,7
53,7/29/2025 0:17,ModernHR,Multiple login failures occurred on user 9052 ...,Security Alert,bert,7
72,2/3/2025 5:35,AnalyticsEngine,User 7153 made multiple incorrect login attempts,Security Alert,bert,7
74,5/9/2025 21:18,BillingSystem,User 8300 made multiple incorrect login attempts,Security Alert,bert,7
113,9/3/2025 2:46,ThirdPartyAPI,Multiple login failures were detected for user...,Security Alert,bert,7
130,12/21/2025 19:36,ThirdPartyAPI,Multiple failed login attempts were reported f...,Security Alert,bert,7
168,5/19/2025 7:56,ModernHR,Multiple incorrect login attempts were made by...,Security Alert,bert,7
173,11/30/2025 2:29,AnalyticsEngine,User 9167 experienced repeated login failures,Security Alert,bert,7
295,8/18/2025 6:57,AnalyticsEngine,Repeated failed login attempts occurred for us...,Security Alert,bert,7
318,12/28/2025 2:39,AnalyticsEngine,User 9131 made multiple failed login attempts,Security Alert,bert,7


In [45]:
#sort clusters by number of records in it. print 5 logs that belong to each cluster.
df.groupby('cluster').size().sort_values(ascending=False)





cluster
0      1017
5       147
11      100
13       86
7        60
       ... 
102       1
103       1
105       1
106       1
135       1
Length: 136, dtype: int64